# Setup and diagnostics

Run this notebook once before Lecture 01. It checks the **downloaded course folder**, the required software versions, the authentic OpenAssistant data files, and the device that will run inference or training.

**Recommended environment:** run this course on **Palmetto 2 with a CUDA GPU**. The GPU keeps model inference and LoRA training practical and is required for the QLoRA extension. You can still run Lectures 01–02 on a local CPU, but generation and training will take longer.

This notebook requires internet access the first time it runs. Hugging Face stores downloaded files in a local cache, so later runs reuse them. If a required package is missing, stop here, follow the environment instructions in the course `README.md`, and then restart JupyterLab with the Hugging Face course kernel.

## 1. Locate the course without depending on the rest of the repository

A notebook may be launched from the course directory or from `notebooks/`. The helper below walks upward until it finds `course_helpers.py` and `data/`. We write it out here first because this is a useful pattern in any standalone notebook project. After checking it once, we import the maintained copy from the module.

In [ ]:
from pathlib import Path

def find_course_root(start=None):
    start_path = Path(start or Path.cwd()).resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / "course_helpers.py").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Start JupyterLab inside the downloaded course folder.")

COURSE_ROOT = find_course_root()
print("Course folder:", COURSE_ROOT)

The printed path should be the downloaded course folder containing `course_helpers.py`, `data/`, and `notebooks/`. If any of those are missing, stop here: relative data and adapter paths would otherwise be misleading. The same function now lives in `course_helpers.py`, where later notebooks can reuse it without hiding its logic.

In [ ]:
import sys
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from course_helpers import find_course_root, read_jsonl, sha256_file
assert find_course_root() == COURSE_ROOT

## 2. Verify package versions and hardware

Package compatibility matters because Transformers, TRL, and PEFT exchange tokenizer, model, dataset, and configuration objects. A successful import is not enough; record the versions that produced your results. Before starting model inference or training on Palmetto 2, confirm that your job has been assigned a CUDA GPU.

In [ ]:
import platform
import shutil
from importlib.metadata import PackageNotFoundError, version

expected = {
    "torch": "2.8.0",
    "torchvision": "0.23.0",
    "transformers": "5.17.0",
    "datasets": "5.0.1",
    "accelerate": "1.15.0",
    "peft": "0.21.2",
    "trl": "1.14.1",
    "qwen-omni-utils": "0.0.9",
    "soundfile": "0.13.1",
}

def installed_version(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return "not installed"

installed = {name: installed_version(name) for name in expected}
mismatches = {
    name: (installed[name], wanted)
    for name, wanted in expected.items()
    if installed[name].split("+")[0] != wanted
}
print("Python:", platform.python_version())
for name, wanted in expected.items():
    status = "PASS" if name not in mismatches else "UPDATE NEEDED"
    print(f"{name:12} installed={installed[name]:12} expected={wanted:8} {status}")

stack_ready = not mismatches
print("Compatible package set:", stack_ready)
if not stack_ready:
    print("Install every pin in:", COURSE_ROOT / "requirements.txt")
    print("Then restart the kernel and rerun this cell.")
else:
    import torch, torchvision, transformers, datasets, accelerate, peft, trl
    import qwen_omni_utils, soundfile
    from peft import LoraConfig
    from trl import SFTTrainer
    from torchvision.ops import nms

    print("Core imports: PASS")
    print("ffmpeg available:", shutil.which("ffmpeg") is not None)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("BF16 supported:", torch.cuda.is_bf16_supported())
    qwen35_ready = hasattr(transformers, "Qwen3_5ForConditionalGeneration")
    print("Qwen3.5 architecture available:", qwen35_ready)

Every package row should end in `PASS`, followed by `Core imports: PASS`. Checking installed metadata before importing PEFT keeps a mixed environment from hiding the useful diagnosis behind a long `BloomPreTrainedModel` traceback. If any row says `UPDATE NEEDED`, install the entire requirements file rather than upgrading one library, restart the kernel, and rerun the cell. `Qwen3.5 architecture available` must be `True` before Lecture 01's Qwen3.5 lab. On Palmetto 2, `CUDA available` should also be `True` and the next line should name the assigned GPU.

## 3. Verify the authentic classroom data

The course packages 2,000 training conversations and 250 held-out conversations derived from `OpenAssistant/oasst1`. The split unit is the original message-tree ID, not an extracted row.

`data/manifests/classroom.json` is the dataset's **packing list**, not another training file. It records where the data came from, which source revision and license apply, how it was split, how many rows should exist, and the expected byte count and SHA-256 fingerprint of each JSONL file. Matching hashes confirm that your copy is the same unmodified classroom dataset used in this course.

In [ ]:
import json

manifest_path = COURSE_ROOT / "data" / "manifests" / "classroom.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
for record in manifest["files"]:
    relative = Path(record["path"])
    file_path = COURSE_ROOT / relative
    actual = sha256_file(file_path)
    size_matches = file_path.stat().st_size == record["bytes"]
    print(relative, "size:", size_matches, "sha256:", actual == record["sha256"])

train_rows = read_jsonl(COURSE_ROOT / "data" / "classroom" / "sft_train.jsonl")
eval_rows = read_jsonl(COURSE_ROOT / "data" / "evaluation" / "sft_evaluation.jsonl")
print("rows:", len(train_rows), len(eval_rows))
print("first roles:", [m["role"] for m in train_rows[0]["messages"]])

Each file should report `size: True` and `sha256: True`, followed by row counts `2000 250`. The role sequence should end with `assistant`. A failed size check suggests truncation; a failed hash with the same size suggests that content changed. In either case, re-download the course rather than training on an unknown file.

## 4. Download the classroom tokenizer and configuration

This cell always requests the tokenizer and configuration at the pinned revision. Internet access is required on the first run. Later runs normally read the same files from the Hugging Face cache. The 135M model weights are loaded in Lecture 01 immediately before you use them for real inference.

In [ ]:
from transformers import AutoConfig, AutoTokenizer
from course_helpers import CLASSROOM_MODEL, CLASSROOM_REVISION

tokenizer = AutoTokenizer.from_pretrained(CLASSROOM_MODEL, revision=CLASSROOM_REVISION)
config = AutoConfig.from_pretrained(CLASSROOM_MODEL, revision=CLASSROOM_REVISION)
print("Cached model files are ready.")
print(type(tokenizer).__name__, config.model_type, config.num_hidden_layers)

You should see `Cached model files are ready`, followed by the tokenizer class, model type, and number of layers. If this cell reports a connection error, confirm that the compute node has internet access and try again; do not remove the pinned revision.

## Ready for Lecture 01

You have now checked four separate contracts: folder layout, software versions, hardware capabilities, and data identity. Keep this notebook with your submission environment information. It contains no API keys and does not depend on another WattAI course.